### 🔄 Rollback Pipeline  — AdventureWorks

**Catálogo:** `catadvwd` | **Plataforma:** Databricks + Delta Lake + Unity Catalog  
**Programa:** G15-Ingeniería de Datos e IA con Databricks | **Julio 2026**

---

## 1. Visión general del pipeline

```
CSV en ADLS Gen2
      │
      ▼
┌─────────────┐     DROP + OVERWRITE      ┌──────────────────────────┐
│   BRONZE    │ ──────────────────────►   │ customers                │
│  Raw Copy   │                           │ productos                │
└─────────────┘                           │ salesorderdetail         │
                                          │ salesorderheader         │
                                          └──────────────────────────┘
      │
      ▼
┌─────────────┐     MERGE (upsert)        ┌──────────────────────────┐
│   SILVER    │ ──────────────────────►   │ customers                │
│  Cleaned    │                           │ products                 │
└─────────────┘                           │ salesorderdetail         │
                                          │ salesorderheader         │
                                          └──────────────────────────┘
      │
      ▼
┌─────────────┐     OVERWRITE completo    ┌──────────────────────────┐
│    GOLD     │ ──────────────────────►   │ dimcustomer  (Dim)       │
│ Star Schema │                           │ dimproduct   (Dim)       │
└─────────────┘                           │ factsales    (Fact)      │
                                          └──────────────────────────┘
```

---

## 2. Notebooks del proyecto

| # | Notebook | Capa | Propósito |
|---|---|---|---|
| 3 | `CustomerBronze` | Bronze | Ingesta `Customer.csv` → `bronze.customers` |
| 4 | `ProductBronze` | Bronze | Ingesta `Product.csv` → `bronze.productos` |
| 5 | `SalesOrderDetailBronze` | Bronze | Ingesta `SalesOrderDetail.csv` → `bronze.salesorderdetail` |
| 6 | `SalesOrderHeaderBronze` | Bronze | Ingesta `SalesOrderHeader.csv` → `bronze.salesorderheader` |
| 7 | `SilverLayer` | Silver | Calidad + MERGE → 4 tablas Silver |
| 8 | `GoldLayer` | Gold | Star schema → `dimcustomer`, `dimproduct`, `factsales` |
| — | `AdventureWorks_Medallion_Pipeline` | Bronze+Silver+Gold | Pipeline completo en un solo notebook |
| 9 | `Rollback_Pipeline` | Todas | DROP + UNDROP por tabla y por capa |

> **Orden de ejecución obligatorio:** Bronze → Silver → Gold. Cada capa depende de la anterior.

---

## 3. Pasos detallados por capa

### 3.1 Bronze — Ingesta fiel del origen

**Objetivo:** espejo exacto de los CSV, sin lógica de negocio.

| Paso | Qué hace |
|---|---|
| 1 | Configurar widgets (`storageName`, `catalogName`, `schemasName`) |
| 2 | Construir ruta ADLS: `abfss://bronze@adventureworksltadlsd.dfs.core.windows.net/` |
| 3 | Definir `StructType` explícito (sin `inferSchema`) |
| 4 | `DROP TABLE IF EXISTS` para idempotencia |
| 5 | Leer CSV con esquema fijo, `header=True` |
| 6 | Selección explícita de columnas (contrato de datos) |
| 7 | Agregar columna de auditoría `createdDate = current_timestamp()` |
| 8 | Escribir Delta con `overwrite` → `saveAsTable` |
| 9 | Verificar con `SELECT *` y `DESCRIBE HISTORY` |

**Reglas Bronze:**
- Todo `nullable = True` — Bronze tolera datos incompletos
- `rowguid` sí se carga en Bronze (se descarta en Silver)
- Nombre de tabla en español (`productos`) — se corrige en Silver

---

### 3.2 Silver — Datos limpios y validados

**Objetivo:** aplicar reglas de calidad y cargar con MERGE (upsert / SCD tipo 1).

| Paso | Qué hace |
|---|---|
| 1 | Leer tabla Bronze desde el catálogo |
| 2 | Aplicar filtros de calidad (ver tabla abajo) |
| 3 | Deduplicar por llave de negocio (solo `salesorderdetail`) |
| 4 | Descartar `rowguid` — sin valor analítico en el lakehouse |
| 5 | Imprimir conteo: total vs. válidas vs. descartadas |
| 6 | Ejecutar `MERGE` (upsert) por llave de negocio |

**Reglas de calidad por tabla:**

| Tabla | Regla | Acción si falla |
|---|---|---|
| `customers` | `firstName` y `lastName` no nulos | Descarta fila |
| `products` | `productId` no nulo | Descarta fila |
| `salesorderdetail` | `salesOrderId`, `salesOrderDetailId`, `orderQty`, `productId`, `unitPrice` no nulos + `salesOrderDetailId` único | Descarta fila / conserva más reciente |
| `salesorderheader` | `customerId` no nulo | Descarta fila |

**Estrategia MERGE — SCD tipo 1:**
```python
delta_tabla.alias("destino")
    .merge(df.alias("origen"), "destino.id = origen.id")
    .whenMatchedUpdateAll()     # registro existe → actualizar
    .whenNotMatchedInsertAll()  # registro nuevo → insertar
    .execute()
```

---

### 3.3 Gold — Modelo dimensional (Star Schema)

**Objetivo:** estructura analítica lista para dashboards y reportería.

| Paso | Qué hace |
|---|---|
| 1 | Leer 4 tablas Silver desde el catálogo |
| 2 | Seleccionar solo columnas analíticas (sin IDs técnicos, sin auditoría) |
| 3 | Construir `dimcustomer` (10 columnas) desde `silver.customers` |
| 4 | Construir `dimproduct` (13 columnas) desde `silver.products` |
| 5 | `INNER JOIN` entre `salesorderdetail` y `salesorderheader` |
| 6 | Construir `factsales` (15 columnas: llaves + métricas + atributos de encabezado) |
| 7 | Escribir las 3 tablas con `overwrite` completo |
| 8 | Verificar conteos finales |

**Grano de `factsales`: línea de detalle (una fila por producto vendido)**

> Una orden puede tener N productos → el grano de línea permite conectar limpiamente `dimcustomer` y `dimproduct` al fact table. Si el grano fuera la orden completa, no habría un solo `productId` válido por fila.

**Medidas semi-aditivas — cómo sumar correctamente:**

```sql
-- ❌ INCORRECTO: totalDue se repite por cada línea de la misma orden
SELECT SUM(totalDue) FROM catadvwd.gold.factsales;

-- ✅ CORRECTO: deduplicar por orden primero
SELECT SUM(totalDue) FROM (
  SELECT DISTINCT salesOrderId, totalDue
  FROM catadvwd.gold.factsales
);

-- ✅ SIEMPRE CORRECTO: lineTotal es aditivo al grano del fact
SELECT SUM(lineTotal) FROM catadvwd.gold.factsales;
```

---

## 4. Rollback — DROP TABLE vs UNDROP TABLE

Esta es la diferencia más importante a entender antes de tocar el notebook `9_Rollback_Pipeline`.

### 4.1 DROP TABLE

```sql
DROP TABLE IF EXISTS catadvwd.bronze.customers;
```

| Aspecto | Detalle |
|---|---|
| **Qué borra** | El registro del metastore **+** los archivos físicos en ADLS |
| **Es reversible** | ✅ Sí, pero solo dentro del período de retención (7 días por defecto) |
| **Efecto inmediato** | La tabla desaparece del catálogo al instante |
| **Cuándo usarlo** | Quieres reingestar desde cero / limpiar el entorno |
| **`IF EXISTS`** | Siempre usarlo — evita errores si la tabla ya no existe |

> ⚠️ En **managed tables** (como las de este pipeline), `DROP TABLE` borra los archivos físicos. En **external tables**, solo borra el registro del catálogo — los archivos sobreviven.

### 4.2 UNDROP TABLE

```sql
UNDROP TABLE catadvwd.bronze.customers;
```

| Aspecto | Detalle |
|---|---|
| **Qué recupera** | El registro del metastore **+** los archivos físicos en ADLS |
| **Requisito** | Unity Catalog habilitado + período de retención vigente (máx. 7 días) |
| **Qué pasa si vence el plazo** | Recuperación imposible — hay que reconstruir desde la capa anterior |
| **Cuándo usarlo** | Ejecutaste un DROP por error y necesitas recuperar la tabla |
| **No aplica con PURGE** | `DROP TABLE ... PURGE` borra de forma inmediata e irrecuperable |

### 4.3 Tabla comparativa

| Característica | `DROP TABLE` | `UNDROP TABLE` |
|---|---|---|
| Dirección de la operación | Elimina | Recupera |
| Borra metastore | ✅ Sí | ↩️ Lo restaura |
| Borra archivos físicos | ✅ Sí (managed) | ↩️ Los reactiva |
| Reversible | ✅ Con UNDROP (7 días) | N/A |
| Requiere Unity Catalog | ✅ Sí | ✅ Sí |
| Equivalente SQL Server | `DROP TABLE` | No existe en SQL Server nativo |
| Disponible en Hive Metastore | ✅ Sí | ❌ No (solo Unity Catalog) |

### 4.4 Orden correcto de rollback

```
Orden CORRECTO (de afuera hacia adentro):
  Gold     ← borrar/recuperar primero
  Silver   ← borrar/recuperar segundo
  Bronze   ← borrar/recuperar último

Orden INCORRECTO:
  Bronze   ← ❌ Si lo borras primero, Silver y Gold
  Silver      quedan huérfanos sin fuente para
  Gold        reconstruirse
```

### 4.5 Log de auditoría en el notebook

Cada operación DROP o UNDROP registra automáticamente un timestamp:

```
🗑️  [2026-07-13 10:45:32]  DROP ejecutado sobre: catadvwd.bronze.customers
♻️  [2026-07-13 10:47:01]  UNDROP ejecutado sobre: catadvwd.silver.products
```

Esto es el equivalente de `GETDATE()` en SQL Server — `current_timestamp()` en PySpark — aplicado como log de auditoría de operaciones destructivas.

---

## 5. Mapeo de tipos de dato — referencia rápida

| Tipo SQL Server | Tipo PySpark | Columnas de ejemplo |
|---|---|---|
| `int` | `IntegerType()` | `customerId`, `productId`, `salesOrderId` |
| `smallint` | `ShortType()` | `orderQty` |
| `decimal(10,2)` | `DecimalType(10,2)` | `standardCost`, `listPrice` |
| `decimal(13,2)` | `DecimalType(13,2)` | `unitPrice`, `lineTotal` |
| `decimal(10,3)` | `DecimalType(10,3)` | `weight`, `subTotal`, `totalDue` |
| `datetime` | `TimestampType()` | `orderDate`, `modifiedDate`, `sellStartDate` |
| `varchar(n)` | `StringType()` | `firstName`, `productName`, `salesOrderNumber` |
| `uniqueidentifier` | `StringType()` | `rowguid` (no se mapea a Silver/Gold) |
| `bit` | `IntegerType()` | `onlineOrderFlag` |

---

## 6. Gobernanza — decisiones clave

| Marco | Decisión aplicada |
|---|---|
| **DAMA-DMBOK** | Bronze = fidelidad, Silver = calidad, Gold = valor analítico |
| **ISO 8000** | Esquema explícito en Bronze (accuracy), reglas de completitud en Silver |
| **GDPR** | `emailAddress` y `phone` en `dimcustomer` son datos personales — acceso restringido por rol en Unity Catalog |
| **Kimball** | Star schema en Gold: dimensiones + fact table con grano declarado |

---

**Autora:** Karina, Arquitecta de Datos Sénior  
**Versión:** 1.0 | Julio 2026

In [0]:
from pyspark.sql.functions import current_timestamp
from datetime import datetime
 
def log_operacion(operacion, tabla_completa):
    """
    Registra en consola la operación ejecutada con su timestamp.
 
    Parámetros:
        operacion (str): 'DROP' o 'UNDROP'
        tabla_completa (str): nombre completo catálogo.esquema.tabla
    """
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    icono = "🗑️" if operacion == "DROP" else "♻️"
    print(f"{icono}  [{ts}]  {operacion} ejecutado sobre: {tabla_completa}")

In [0]:
# Nota: esta tabla se llama `productos` en Bronze (español). En Silver se renombró a `products` (inglés).
spark.sql("DROP TABLE IF EXISTS catadvwd.bronze.customers")
log_operacion("DROP", "catadvwd.bronze.customers")

In [0]:
spark.sql("DROP TABLE IF EXISTS catadvwd.bronze.productos")
log_operacion("DROP", "catadvwd.bronze.productos")

In [0]:

spark.sql("DROP TABLE IF EXISTS catadvwd.bronze.salesorderdetail")
log_operacion("DROP", "catadvwd.bronze.salesorderdetail")

In [0]:

spark.sql("DROP TABLE IF EXISTS catadvwd.bronze.salesorderheader")
log_operacion("DROP", "catadvwd.bronze.salesorderheader")
 

In [0]:

spark.sql("DROP TABLE IF EXISTS catadvwd.silver.customers")
log_operacion("DROP", "catadvwd.silver.customers")

In [0]:

spark.sql("DROP TABLE IF EXISTS catadvwd.silver.products")
log_operacion("DROP", "catadvwd.silver.products")

In [0]:

spark.sql("DROP TABLE IF EXISTS catadvwd.silver.salesorderdetail")
log_operacion("DROP", "catadvwd.silver.salesorderdetail")
 

In [0]:
 
spark.sql("DROP TABLE IF EXISTS catadvwd.silver.salesorderheader")
log_operacion("DROP", "catadvwd.silver.salesorderheader")

In [0]:
spark.sql("DROP TABLE IF EXISTS catadvwd.gold.dimcustomer")
log_operacion("DROP", "catadvwd.gold.dimcustomer")

In [0]:

spark.sql("DROP TABLE IF EXISTS catadvwd.gold.dimproduct")
log_operacion("DROP", "catadvwd.gold.dimproduct")

In [0]:
spark.sql("DROP TABLE IF EXISTS catadvwd.gold.factsales")
log_operacion("DROP", "catadvwd.gold.factsales")

### SECCIÓN 4 — UNDROP TABLE: Bronze
 `UNDROP TABLE` recupera una tabla eliminada dentro del período de retención de Unity Catalog (por defecto **7 días**).
 Nota: si `UNDROP` falla con "Table not found in deleted tables", el período de retención venció. La única opción es reconstruir la tabla volviendo a correr el notebook de Bronze correspondiente.
 

In [0]:
spark.sql("UNDROP TABLE catadvwd.bronze.customers")
log_operacion("UNDROP", "catadvwd.bronze.customers")

In [0]:
spark.sql("UNDROP TABLE catadvwd.bronze.salesorderdetail")
log_operacion("UNDROP", "catadvwd.bronze.salesorderdetail")

In [0]:

spark.sql("UNDROP TABLE catadvwd.bronze.salesorderheader")
log_operacion("UNDROP", "catadvwd.bronze.salesorderheader")

In [0]:
spark.sql("UNDROP TABLE catadvwd.silver.customers")
log_operacion("UNDROP", "catadvwd.silver.customers")

In [0]:
spark.sql("UNDROP TABLE catadvwd.silver.products")
log_operacion("UNDROP", "catadvwd.silver.products")

In [0]:

spark.sql("UNDROP TABLE catadvwd.silver.salesorderdetail")
log_operacion("UNDROP", "catadvwd.silver.salesorderdetail")
 

In [0]:
spark.sql("UNDROP TABLE catadvwd.silver.salesorderheader")
log_operacion("UNDROP", "catadvwd.silver.salesorderheader")

In [0]:
spark.sql("UNDROP TABLE catadvwd.gold.dimcustomer")
log_operacion("UNDROP", "catadvwd.gold.dimcustomer")

In [0]:
spark.sql("UNDROP TABLE catadvwd.gold.dimproduct")
log_operacion("UNDROP", "catadvwd.gold.dimproduct")

In [0]:
spark.sql("UNDROP TABLE catadvwd.gold.factsales")
log_operacion("UNDROP", "catadvwd.gold.factsales")

### SECCIÓN 7 — Verificación del estado actual

Primero hay que orre esta celda en cualquier momento para ver qué tablas existen y cuántas filas tienen.
El timestamp al inicio indica exactamente cuándo se ejecutó el reporte en tiempo actual .

In [0]:
capas = {
    "Bronze": {"schema": "bronze", "tablas": ["customers", "productos", "salesorderdetail", "salesorderheader"]},
    "Silver": {"schema": "silver", "tablas": ["customers", "products", "salesorderdetail", "salesorderheader"]},
    "Gold":   {"schema": "gold",   "tablas": ["dimcustomer", "dimproduct", "factsales"]}
}
 
ts_reporte = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
 
print("=" * 60)
print(f"{'REPORTE DE ESTADO — catadvwd':^60}")
print(f"{'Generado: ' + ts_reporte:^60}")
print("=" * 60)
 
for capa, info in capas.items():
    schema = info["schema"]
    print(f"\n📂 {capa.upper()} — catadvwd.{schema}")
    print("-" * 50)
    for tabla in info["tablas"]:
        nombre_completo = f"catadvwd.{schema}.{tabla}"
        existe = spark.catalog.tableExists(nombre_completo)
        icono = "✅" if existe else "❌"
        estado = "existe     " if existe else "eliminada  "
        filas = ""
        if existe:
            try:
                n = spark.table(nombre_completo).count()
                filas = f"| {n:,} filas"
            except Exception:
                filas = "| error al contar"
        print(f"  {icono}  {tabla:<32} {estado} {filas}")
 
print("\n" + "=" * 60)
print(f"  Fin del reporte — {ts_reporte}")
print("=" * 60)
 
}